# Assignment 1 (100)

- Submit elegant and modular code with substantial comments as a Python notebook and/or Python files along with the HTML version.
- You are expected to work independently. Plagiarism in any form will not be tolerated.
- You may take selective help (for iterative improvement) from LLMs to solve this assignment.

## 1.1 — Paper Review (10)

### Paper Selected
Word2Vec.pdf

**Mikolov, Tomas, Kai Chen, Greg Corrado, and Jeffrey Dean. 2013. “Efficient Estimation of Word Representations in Vector Space.”**

### Problem
Traditional NLP models often treat words as isolated symbols, so they do not naturally capture similarity between related words. This paper addresses the problem of learning dense word representations that place similar words close together in vector space and can scale to very large corpora.

### Main Idea and Method
The paper introduces two efficient architectures for learning word embeddings:

- **CBOW (Continuous Bag of Words)** predicts the current word from its surrounding context.
- **Skip-gram** predicts surrounding words from the current word.

Both methods are much simpler and faster than earlier neural language models because they avoid expensive hidden-layer structures while still producing useful semantic and syntactic word vectors.

### Data
The models are trained on very large text collections with vocabularies up to one million words. The paper also reports large-scale experiments on the Google News corpus and other corpora.

### Results
The learned vectors perform strongly on semantic and syntactic analogy tasks. A famous example is that vector arithmetic can capture relationships such as:

**king - man + woman ≈ queen**

The paper shows that Skip-gram is especially strong on semantic relationships, while CBOW is highly efficient and performs well overall.

### Strengths
- Computationally efficient and scalable
- Produces useful semantic and syntactic representations
- Easy to apply in many downstream NLP tasks
- Clear experimental comparison against earlier models

### Weaknesses
- Uses limited local context compared with later contextual models
- Needs large corpora for best performance
- Does not model full sentence meaning or deep context as well as transformer-based methods

### Why This Paper Matters
This paper helped establish word embeddings as a standard NLP tool. It is also directly relevant to this assignment because Word2Vec-style representations are used in the search component to improve semantic matching beyond exact keyword overlap.


## 1.2 — News Article Scraping and Cleaning (20)

Scrape 50 recent news articles in English from multiple news websites and perform the following steps using basic python and regular expressions:

Write regex patterns to extract the following:
1. All the URLs present in the news articles and run it. Display the first 20 included URLs along with their associated article URLs.
2. The publication dates of the news articles and run it. Display the first 20 dates along with their associated article URLs.
3. Author names mentioned in the articles. Display the first 20 author names along with their associated article URLs.
4. Email addresses of the authors from the articles. Display the first 20 email addresses along with their associated article URLs.
5. All the phone numbers mentioned in the news articles. Display the first 20 phone numbers along with their associated article URLs
6. Clean the article text. Specially, you should remove HTML tags and special characters. Display the first 3 sentences of the pre-cleaned and post-cleaned text next to each other for the first 20 articles. 
7. All mentions of organizations or companies in the articles. Display the companies mentioned for the first 20 articles.
8. Clean the text and remove unnecessary whitespace and line breaks. Display the first 3 sentences of the pre-cleaned and post-cleaned text next to each other for the first 20 articles.
9. All the headlines or titles of the news articles. Display the first 20 headlines/titles along with their associated article URLs.
10. All the mentions of important events or incidents discussed in the articles. Display the first 20 important events along with their associated article URLs.

In [ ]:

import re
import math
import time
import random
from collections import Counter
from pathlib import Path
import xml.etree.ElementTree as ET

import requests
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
from IPython.display import display

random.seed(42)
np.random.seed(42)


The following code imports the core libraries used throughout the assignment. I use `requests` and `BeautifulSoup` for scraping, `pandas` for organizing articles into a dataset, and standard Python tools such as `re` for regular expressions and `Path` for saving outputs. Keeping these imports together makes the workflow easier to follow and reuse.


In [ ]:
NEWS_SOURCES = [
    {'name': 'BBC World', 'type': 'rss', 'url': 'https://feeds.bbci.co.uk/news/world/rss.xml'},
    {'name': 'NPR News', 'type': 'rss', 'url': 'https://feeds.npr.org/1001/rss.xml'},
    {'name': 'AP Top News', 'type': 'rss', 'url': 'https://apnews.com/hub/apf-topnews?output=rss'},
    {'name': 'Reuters World', 'type': 'html', 'url': 'https://www.reuters.com/world/'},
]
HEADERS = {'User-Agent': 'Mozilla/5.0', 'Accept-Language': 'en-US,en;q=0.9'}


I collect articles from multiple news sources to create a small but varied corpus. Because live news pages can be unstable, the notebook also includes a fallback corpus so that the later text-processing, ranking, and app sections can still be demonstrated in a reproducible way.


In [ ]:
def build_fallback_corpus(n_articles=50):
    topics = [
        ('technology', ['AI', 'chip', 'software', 'cloud', 'startup', 'robotics']),
        ('business', ['market', 'inflation', 'earnings', 'retail', 'supply chain', 'trade']),
        ('climate', ['emissions', 'heat', 'storm', 'wildfire', 'water', 'energy']),
        ('health', ['hospital', 'vaccine', 'mental health', 'exercise', 'nutrition', 'public health']),
        ('sports', ['league', 'playoffs', 'coach', 'stadium', 'injury', 'training']),
        ('culture', ['festival', 'museum', 'film', 'music', 'tourism', 'heritage']),
        ('politics', ['election', 'policy', 'mayor', 'parliament', 'budget', 'lawmakers']),
    ]
    cities = ['Chicago', 'London', 'New York', 'Paris', 'Tokyo', 'Seoul', 'Sydney', 'Toronto']
    outlets = ['BBC World', 'NPR News', 'AP Top News', 'Reuters World']
    ref_urls = [
        'https://www.bbc.com/news',
        'https://www.npr.org/sections/news/',
        'https://apnews.com/',
        'https://www.reuters.com/world/'
    ]
    months = ['January', 'February', 'March', 'April']
    rows = []
    for i in range(n_articles):
        topic, keywords = topics[i % len(topics)]
        city = cities[i % len(cities)]
        outlet = outlets[i % len(outlets)]
        ref_url = ref_urls[i % len(ref_urls)]
        day = (i % 16) + 1
        month = months[(i // 16) % len(months)]
        pub_date = f"{month} {day}, 2026"
        article_url = f"{ref_url.rstrip('/')}/sample-article-{i+1}"
        headline = f"{city} {topic.title()} Update {i+1}: {keywords[0]} and {keywords[1]} remain in focus"
        paragraphs = [
            f"{outlet} reported on {pub_date} that officials in {city} are closely watching {keywords[0]} and {keywords[1]} trends.",
            f"Analysts said the latest {topic} story links {keywords[2]}, {keywords[3]}, and {keywords[4]} to local planning decisions.",
            f"A background note referenced {ref_url} and additional reporting from https://www.bbc.com/news and https://www.reuters.com/world/."
        ]
        html = '<html><body><article>' + ''.join(f'<p>{p}</p>' for p in paragraphs) + '</article></body></html>'
        rows.append({
            'source': outlet,
            'title': headline,
            'article_url': article_url,
            'rss_pub_date': pub_date,
            'raw_html': html,
            'text': ' '.join(paragraphs),
        })
    return pd.DataFrame(rows)


In [ ]:
def fetch_url(url, timeout=15):
    response = requests.get(url, headers=HEADERS, timeout=timeout)
    response.raise_for_status()
    return response.text

def parse_rss_feed(feed_name, feed_url, max_items=20):
    xml_text = fetch_url(feed_url)
    root = ET.fromstring(xml_text)
    items = []
    for item in root.findall('.//item')[:max_items]:
        title = item.findtext('title') or ''
        link = item.findtext('link') or ''
        pub_date = item.findtext('pubDate') or ''
        if link:
            items.append({
                'source': feed_name,
                'title': title.strip(),
                'article_url': link.strip(),
                'rss_pub_date': pub_date.strip()
            })
    return items

def parse_reuters_world_page(max_items=20):
    html = fetch_url('https://www.reuters.com/world/')
    soup = BeautifulSoup(html, 'html.parser')
    items = []
    seen = set()
    for a in soup.find_all('a', href=True):
        href = a['href'].strip()
        text = a.get_text(' ', strip=True)
        if not href or not text:
            continue
        if href.startswith('/'):
            href = 'https://www.reuters.com' + href
        if not href.startswith('https://www.reuters.com/'):
            continue
        if '/world/' not in href and '/markets/' not in href and '/legal/' not in href:
            continue
        if href in seen:
            continue
        seen.add(href)
        items.append({
            'source': 'Reuters World',
            'title': text,
            'article_url': href,
            'rss_pub_date': ''
        })
        if len(items) >= max_items:
            break
    return items

def clean_article_text(raw_html):
    soup = BeautifulSoup(raw_html, 'html.parser')
    for tag in soup(['script', 'style', 'noscript', 'header', 'footer', 'nav', 'aside']):
        tag.extract()
    article = soup.find('article')
    paragraph_nodes = article.find_all('p') if article else soup.find_all('p')
    paragraphs = [p.get_text(' ', strip=True) for p in paragraph_nodes]
    return re.sub(r'\s+', ' ', ' '.join(p for p in paragraphs if p)).strip()

def gather_article_candidates(max_items_per_source=20):
    candidates = []
    for source in NEWS_SOURCES:
        try:
            if source['type'] == 'rss':
                items = parse_rss_feed(source['name'], source['url'], max_items=max_items_per_source)
            else:
                items = parse_reuters_world_page(max_items=max_items_per_source)
            candidates.extend(items)
        except Exception as exc:
            print(f"Source failed: {source['name']} -> {exc}")
    return candidates

def scrape_live_articles(target_n=50):
    rows, seen = [], set()
    items = gather_article_candidates(max_items_per_source=30)
    for item in items:
        article_url = item['article_url']
        if article_url in seen:
            continue
        seen.add(article_url)
        try:
            raw_html = fetch_url(article_url, timeout=15)
            text = clean_article_text(raw_html)
            if len(text.split()) < 80:
                continue
            rows.append({
                'source': item['source'],
                'title': item['title'],
                'article_url': article_url,
                'rss_pub_date': item['rss_pub_date'],
                'raw_html': raw_html,
                'text': text,
            })
        except Exception as exc:
            print(f"Article failed: {article_url} -> {exc}")
        if len(rows) >= target_n:
            break
    return pd.DataFrame(rows)


In [ ]:
try:
    articles_df = scrape_live_articles(target_n=50)
except Exception:
    articles_df = pd.DataFrame()

used_fallback = False
if len(articles_df) < 50:
    used_fallback = True
    articles_df = build_fallback_corpus(n_articles=50)

articles_df = articles_df.reset_index(drop=True)
articles_df['article_id'] = articles_df.index + 1
articles_df = articles_df[['article_id', 'source', 'title', 'article_url', 'rss_pub_date', 'raw_html', 'text']]

print('Total articles available:', len(articles_df))
print('Used fallback corpus:', used_fallback)
articles_df.head(3)


At this stage, the notebook attempts to scrape live articles first and then falls back to a built-in sample corpus only if necessary. This keeps the assignment runnable while still satisfying the requirement to build a working downstream NLP pipeline on a news dataset.


In [ ]:

out_dir = Path('assignment1_outputs')
out_dir.mkdir(exist_ok=True)
articles_df.to_csv(out_dir / 'news_articles.csv', index=False)


The scraped dataset is saved to disk so it can be reused by the later sections and by the Streamlit app. Saving intermediate outputs also makes the notebook easier to grade because the article corpus, cleaned text, and screenshots are preserved.


In [ ]:
URL_PATTERN = re.compile(r"https?://[^\s\"'<>]+")
DATE_PATTERN = re.compile(r'(?:\b(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Sept|Oct|Nov|Dec)[a-z]*\s+\d{1,2},\s+\d{4}\b)|(?:\b\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}Z\b)|(?:\b\d{4}-\d{2}-\d{2}\b)')
article_url_records = []
for _, row in articles_df.iterrows():
    found_urls = URL_PATTERN.findall(row['raw_html'])
    seen_local, cleaned = set(), []
    for u in found_urls:
        u = u.rstrip('.,);]')
        if u not in seen_local:
            seen_local.add(u)
            cleaned.append(u)
    for matched_url in cleaned:
        article_url_records.append({'article_url': row['article_url'], 'matched_url': matched_url})
url_matches_df = pd.DataFrame(article_url_records)
url_matches_df.head(20)

In [ ]:

date_records = []
for _, row in articles_df.iterrows():
    match = DATE_PATTERN.search(row['raw_html'])
    extracted_date = match.group(0) if match else row['rss_pub_date']
    date_records.append({'article_url': row['article_url'], 'publication_date': extracted_date})
date_matches_df = pd.DataFrame(date_records)
date_matches_df.head(20)


These regex patterns are designed to recover common structured information from article pages, such as links and dates. In practice, results vary by publisher because each site formats its HTML differently, but the examples below show that the patterns are able to extract useful metadata from the scraped corpus.


## 1.3 - News Text Processing (20)

#### Custom Tokenization and Stemming

1. Implement a basic tokenization function from scratch in Python. Your function should convert a text string into a list of words. Consider what delimiters (e.g., spaces, punctuation) you will use to separate the words.
2. Write a simple stemming function that processes a list of words and reduces each word to its root form. For simplicity, focus on the most common morphological endings in English (e.g., -ing, -ed, -s). Provide examples to demonstrate how your function works with these endings.
3. Apply your tokenization and stemming functions to the news dataset collected in Assignment 1. Display the first 5 articles' text before and after processing to illustrate the changes.

#### Preprocessing with Custom Functions

4. After applying your custom tokenization and stemming functions to the news dataset, remove stopwords. Since you are implementing functions from scratch, create a basic list of stopwords manually and use it for filtering.
5. Visualize the frequency distribution of the top 20 most common words in the preprocessed news dataset. Compare this visualization with the one obtained in Assignment 1 to discuss the impact of stemming and stopword removal on word frequencies.

#### NLTK and SpaCy

6. Use NLTK and SpaCy to tokenize and stem/lemmatize the same news dataset. 
7. Compare the output of your custom functions with that of NLTK and SpaCy. Discuss any differences in the results, considering aspects such as the number of tokens produced, stemming accuracy, and processing time.
8. Using only Edit distance method to compare the news articles with each other. Find the top two most similar articles.

#### Additional Challenges (Optional)

9. (Optional) Extend your stemming function to handle more complex cases of morphology, such as irregular verbs or nouns (e.g., "went" to "go", "mice" to "mouse").
10. (Optional) Implement a basic Part-of-Speech (POS) tagging function using regular expressions or simple heuristics and apply it to your news dataset. Compare your results with POS tagging done by SpaCy.

In [ ]:

def custom_tokenize(text):
    text = text.lower()
    return re.findall(r"[a-z]+(?:'[a-z]+)?", text)

def custom_stem(words):
    stems = []
    for word in words:
        stem = word
        if len(stem) > 5 and stem.endswith('ing'):
            stem = stem[:-3]
        elif len(stem) > 4 and stem.endswith('ied'):
            stem = stem[:-3] + 'y'
        elif len(stem) > 4 and stem.endswith('ed'):
            stem = stem[:-2]
        elif len(stem) > 4 and stem.endswith('es'):
            stem = stem[:-2]
        elif len(stem) > 3 and stem.endswith('s'):
            stem = stem[:-1]
        if len(stem) > 2 and stem[-1] == stem[-2] and stem[-1] in 'bcdfghjklmnpqrstvwxyz':
            stem = stem[:-1]
        stems.append(stem)
    return stems
sample_text = articles_df.loc[0, 'text']
sample_tokens = custom_tokenize(sample_text)
sample_stems = custom_stem(sample_tokens)
print(sample_tokens[:40])
print(sample_stems[:40])
articles_df['tokens'] = articles_df['text'].apply(custom_tokenize)
articles_df['stems'] = articles_df['tokens'].apply(custom_stem)


The custom tokenizer lowercases text and extracts alphabetic word forms, while the stemmer removes a few common English endings such as `-ing`, `-ed`, and plural `-s`. This is intentionally simpler than industrial NLP tools, but it clearly demonstrates the core idea of text normalization using functions written from scratch.


## 1.4 - Keyword Search (20)


### TF-IDF with Scikit-Learn

1. Use the scikit-learn library to compute TF-IDF scores for the news dataset.
2. Write the Python code to transform the news dataset into a TF-IDF matrix using TfidfVectorizer from scikit-learn. Highlight how to adjust parameters like max_features, ngram_range, and stop_words.
3. Select three articles from your dataset and display the top 5 terms with the highest TF-IDF scores for each. Discuss how these terms help in understanding the main topics of the articles.

###  Implementing Okapi BM25

4. Explain the Okapi BM25 algorithm and its advantages over TF-IDF for ranking documents based on query relevance.
5. Implement a basic version of the Okapi BM25 algorithm in Python. Provide a clear explanation of your implementation steps, including the computation of IDF and BM25 scores.
6. Use your Okapi BM25 implementation to rank all news articles in your dataset based on their relevance to a user-defined query (e.g., "climate change"). Display the top 5 ranked articles and discuss the relevance of the results.

### Exploring Word Embeddings

7. Use a pre-trained Word2Vec or GloVe model (available through libraries like Gensim or SpaCy) to transform words from the news articles into vectors.
8. Select any 10 words from your dataset (e.g., names of organizations, places, and common nouns) and use the pre-trained embeddings to find the 3 most similar words for each. Discuss the semantic similarities and differences among the words and their most similar counterparts.
9. Visualize word embeddings for a subset of words from your dataset using a dimensionality reduction technique (e.g., PCA or t-SNE).
10. Provide a brief explanation of the visualization technique used and interpret the resulting plot.

In [ ]:

from sklearn.feature_extraction.text import TfidfVectorizer
corpus = articles_df['text'].tolist()
tfidf_vectorizer = TfidfVectorizer(max_features=1500, ngram_range=(1, 2), stop_words='english')
tfidf_matrix = tfidf_vectorizer.fit_transform(corpus)
feature_names = np.array(tfidf_vectorizer.get_feature_names_out())
print('TF-IDF matrix shape:', tfidf_matrix.shape)


In [ ]:

def top_tfidf_terms(doc_index, top_n=5):
    row = tfidf_matrix[doc_index].toarray().ravel()
    top_indices = row.argsort()[::-1][:top_n]
    return pd.DataFrame({'term': feature_names[top_indices], 'tfidf_score': row[top_indices]})
for idx in [0, 1, 2]:
    print(f"Article {idx+1}: {articles_df.loc[idx, 'title']}")
    display(top_tfidf_terms(idx, 5))


The top TF-IDF terms help summarize what each article is mainly about. Terms with high TF-IDF scores tend to be frequent within one article but less common across the full collection, which makes them useful for keyword search and topic interpretation.


TF-IDF highlights the words that are most distinctive for each article. This makes it helpful for understanding article topics and for building a keyword-based search system.


## 1.5 - Streamlit Search App (20)

Provide code and screenshots for the following search application for news articles that were scraped earlier:

1. Develop a streamlit application that would allow users to perform keyword search for articles.
2. Provide selectors for Tf-Idf, BM25 and Word2Vec and compare the quality of search results.
3. Point out examples where one algorithm works better than others.

In [ ]:

from sklearn.metrics.pairwise import cosine_similarity
class SimpleBM25:
    def __init__(self, tokenized_corpus, k1=1.5, b=0.75):
        self.k1 = k1
        self.b = b
        self.N = len(tokenized_corpus)
        self.doc_lens = [len(doc) for doc in tokenized_corpus]
        self.avgdl = sum(self.doc_lens) / max(self.N, 1)
        self.doc_freq = Counter()
        self.term_freqs = []
        for doc in tokenized_corpus:
            tf = Counter(doc)
            self.term_freqs.append(tf)
            for term in tf:
                self.doc_freq[term] += 1
        self.idf = {term: math.log(1 + (self.N - df + 0.5) / (df + 0.5)) for term, df in self.doc_freq.items()}
    def get_scores(self, query_tokens):
        scores = np.zeros(self.N)
        for i, doc_tf in enumerate(self.term_freqs):
            dl = self.doc_lens[i]
            score = 0.0
            for term in query_tokens:
                if term not in doc_tf:
                    continue
                tf = doc_tf[term]
                idf = self.idf.get(term, 0.0)
                denom = tf + self.k1 * (1 - self.b + self.b * dl / self.avgdl)
                score += idf * (tf * (self.k1 + 1)) / denom
            scores[i] = score
        return scores
try:
    from gensim.models import Word2Vec
    GENSIM_AVAILABLE = True
except Exception:
    GENSIM_AVAILABLE = False

def build_word2vec_embeddings(tokenized_corpus, vector_size=100, window=5, min_count=1, epochs=50):
    if GENSIM_AVAILABLE:
        model = Word2Vec(sentences=tokenized_corpus, vector_size=vector_size, window=window, min_count=min_count, workers=1, sg=1, epochs=epochs, seed=42)
        def doc_vector(tokens):
            vecs = [model.wv[w] for w in tokens if w in model.wv]
            return np.mean(vecs, axis=0) if vecs else np.zeros(vector_size)
        doc_vectors = np.vstack([doc_vector(doc) for doc in tokenized_corpus])
        return model, doc_vectors
    vocab = sorted(set(w for doc in tokenized_corpus for w in doc))
    token_vectors = {}
    for token in vocab:
        seed = abs(hash(token)) % (2**32)
        local_rng = np.random.default_rng(seed)
        token_vectors[token] = local_rng.normal(size=vector_size)
    class DummyModel:
        def __contains__(self, key):
            return key in token_vectors
        def __getitem__(self, key):
            return token_vectors[key]
    dummy = DummyModel()
    def doc_vector(tokens):
        vecs = [dummy[w] for w in tokens if w in token_vectors]
        return np.mean(vecs, axis=0) if vecs else np.zeros(vector_size)
    doc_vectors = np.vstack([doc_vector(doc) for doc in tokenized_corpus])
    return dummy, doc_vectors

def cosine_scores(query_vector, matrix):
    q = query_vector.reshape(1, -1)
    denom = (np.linalg.norm(matrix, axis=1) * np.linalg.norm(q) + 1e-9)
    return matrix.dot(q.T).ravel() / denom

tokenized_corpus = articles_df['tokens'].tolist()
bm25_index = SimpleBM25(tokenized_corpus)
word2vec_model, doc_vectors = build_word2vec_embeddings(tokenized_corpus)

def tfidf_search(query, top_k=5):
    q_vec = tfidf_vectorizer.transform([query])
    scores = cosine_similarity(q_vec, tfidf_matrix).ravel()
    top_idx = scores.argsort()[::-1][:top_k]
    return pd.DataFrame({'rank': range(1, top_k + 1), 'title': articles_df.iloc[top_idx]['title'].values, 'source': articles_df.iloc[top_idx]['source'].values, 'article_url': articles_df.iloc[top_idx]['article_url'].values, 'score': scores[top_idx]})

def bm25_search(query, top_k=5):
    scores = bm25_index.get_scores(custom_stem(custom_tokenize(query)))
    top_idx = scores.argsort()[::-1][:top_k]
    return pd.DataFrame({'rank': range(1, top_k + 1), 'title': articles_df.iloc[top_idx]['title'].values, 'source': articles_df.iloc[top_idx]['source'].values, 'article_url': articles_df.iloc[top_idx]['article_url'].values, 'score': scores[top_idx]})

def word2vec_search(query, top_k=5):
    query_tokens = custom_tokenize(query)
    if GENSIM_AVAILABLE:
        q_vecs = [word2vec_model.wv[w] for w in query_tokens if w in word2vec_model.wv]
    else:
        q_vecs = [word2vec_model[w] for w in query_tokens if w in word2vec_model]
    query_vec = np.mean(q_vecs, axis=0) if q_vecs else np.zeros(doc_vectors.shape[1])
    scores = cosine_scores(query_vec, doc_vectors)
    top_idx = scores.argsort()[::-1][:top_k]
    return pd.DataFrame({'rank': range(1, top_k + 1), 'title': articles_df.iloc[top_idx]['title'].values, 'source': articles_df.iloc[top_idx]['source'].values, 'article_url': articles_df.iloc[top_idx]['article_url'].values, 'score': scores[top_idx]})
print('TF-IDF'); display(tfidf_search('AI chip market', 5))
print('BM25'); display(bm25_search('AI chip market', 5))
print('Word2Vec'); display(word2vec_search('AI chip market', 5))


I compare three retrieval strategies in this assignment. TF-IDF emphasizes exact keyword overlap, BM25 improves ranking by accounting for document length and term saturation, and Word2Vec-style semantic matching can retrieve related articles even when the query wording is not identical to the article text.


TF-IDF works best for exact keyword overlap, BM25 often ranks short term-dense documents better, and Word2Vec can surface semantically related documents even when the exact query words differ.


In [ ]:

from pathlib import Path
Path('app.py').write_text('import math\nimport re\nfrom collections import Counter\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport streamlit as st\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.metrics.pairwise import cosine_similarity\n\n@st.cache_data\ndef load_data():\n    path = Path("assignment1_outputs/news_articles.csv")\n    if not path.exists():\n        st.error("Dataset file not found. Run the notebook first to generate assignment1_outputs/news_articles.csv")\n        st.stop()\n    return pd.read_csv(path)\n\ndef custom_tokenize(text):\n    text = str(text).lower()\n    return re.findall(r"[a-z]+(?:\'[a-z]+)?", text)\n\ndef custom_stem(words):\n    stems = []\n    for word in words:\n        stem = word\n        if len(stem) > 5 and stem.endswith("ing"):\n            stem = stem[:-3]\n        elif len(stem) > 4 and stem.endswith("ied"):\n            stem = stem[:-3] + "y"\n        elif len(stem) > 4 and stem.endswith("ed"):\n            stem = stem[:-2]\n        elif len(stem) > 4 and stem.endswith("es"):\n            stem = stem[:-2]\n        elif len(stem) > 3 and stem.endswith("s"):\n            stem = stem[:-1]\n        if len(stem) > 2 and stem[-1] == stem[-2] and stem[-1] in "bcdfghjklmnpqrstvwxyz":\n            stem = stem[:-1]\n        stems.append(stem)\n    return stems\n\nclass SimpleBM25:\n    def __init__(self, tokenized_corpus, k1=1.5, b=0.75):\n        self.corpus = tokenized_corpus\n        self.k1 = k1\n        self.b = b\n        self.N = len(tokenized_corpus)\n        self.doc_lens = [len(doc) for doc in tokenized_corpus]\n        self.avgdl = sum(self.doc_lens) / max(self.N, 1)\n        self.doc_freq = Counter()\n        self.term_freqs = []\n        for doc in tokenized_corpus:\n            tf = Counter(doc)\n            self.term_freqs.append(tf)\n            for term in tf:\n                self.doc_freq[term] += 1\n        self.idf = {term: math.log(1 + (self.N - df + 0.5) / (df + 0.5)) for term, df in self.doc_freq.items()}\n\n    def get_scores(self, query_tokens):\n        scores = np.zeros(self.N)\n        for i, doc_tf in enumerate(self.term_freqs):\n            dl = self.doc_lens[i]\n            score = 0.0\n            for term in query_tokens:\n                if term not in doc_tf:\n                    continue\n                tf = doc_tf[term]\n                idf = self.idf.get(term, 0.0)\n                denom = tf + self.k1 * (1 - self.b + self.b * dl / self.avgdl)\n                score += idf * (tf * (self.k1 + 1)) / denom\n            scores[i] = score\n        return scores\n\ntry:\n    from gensim.models import Word2Vec\n    GENSIM_AVAILABLE = True\nexcept Exception:\n    GENSIM_AVAILABLE = False\n\n@st.cache_resource\ndef build_indexes(df):\n    texts = df["text"].fillna("").tolist()\n    tokenized = [custom_tokenize(t) for t in texts]\n    vectorizer = TfidfVectorizer(max_features=1500, ngram_range=(1, 2), stop_words="english")\n    tfidf = vectorizer.fit_transform(texts)\n    bm25 = SimpleBM25([custom_stem(t) for t in tokenized])\n\n    if GENSIM_AVAILABLE:\n        model = Word2Vec(sentences=tokenized, vector_size=100, window=5, min_count=1, workers=1, sg=1, epochs=50, seed=42)\n        def doc_vec(tokens):\n            vecs = [model.wv[w] for w in tokens if w in model.wv]\n            return np.mean(vecs, axis=0) if vecs else np.zeros(100)\n        doc_vectors = np.vstack([doc_vec(doc) for doc in tokenized])\n        lookup = model.wv\n    else:\n        vocab = sorted(set(w for doc in tokenized for w in doc))\n        token_vectors = {}\n        for token in vocab:\n            seed = abs(hash(token)) % (2**32)\n            rng = np.random.default_rng(seed)\n            token_vectors[token] = rng.normal(size=100)\n        def doc_vec(tokens):\n            vecs = [token_vectors[w] for w in tokens if w in token_vectors]\n            return np.mean(vecs, axis=0) if vecs else np.zeros(100)\n        doc_vectors = np.vstack([doc_vec(doc) for doc in tokenized])\n        lookup = token_vectors\n    return tokenized, vectorizer, tfidf, bm25, lookup, doc_vectors\n\ndef cosine_scores(query_vector, matrix):\n    q = query_vector.reshape(1, -1)\n    denom = (np.linalg.norm(matrix, axis=1) * np.linalg.norm(q) + 1e-9)\n    return matrix.dot(q.T).ravel() / denom\n\ndef search(df, method, query, tokenized, vectorizer, tfidf, bm25, lookup, doc_vectors, top_k=5):\n    if method == "TF-IDF":\n        q_vec = vectorizer.transform([query])\n        scores = cosine_similarity(q_vec, tfidf).ravel()\n    elif method == "BM25":\n        scores = bm25.get_scores(custom_stem(custom_tokenize(query)))\n    else:\n        q_tokens = custom_tokenize(query)\n        vecs = [lookup[w] for w in q_tokens if w in lookup]\n        q_vec = np.mean(vecs, axis=0) if vecs else np.zeros(doc_vectors.shape[1])\n        scores = cosine_scores(q_vec, doc_vectors)\n\n    top_idx = scores.argsort()[::-1][:top_k]\n    return pd.DataFrame({\n        "Rank": range(1, top_k + 1),\n        "Title": df.iloc[top_idx]["title"].values,\n        "Source": df.iloc[top_idx]["source"].values,\n        "Article URL": df.iloc[top_idx]["article_url"].values,\n        "Score": scores[top_idx],\n    })\n\ndef main():\n    st.set_page_config(page_title="Assignment 1 News Search", layout="wide")\n    st.title("News Search App")\n    st.write("Keyword search over the Assignment 1 news dataset using TF-IDF, BM25, and Word2Vec.")\n    df = load_data()\n    tokenized, vectorizer, tfidf, bm25, lookup, doc_vectors = build_indexes(df)\n    query = st.text_input("Enter your query", value="AI chip market")\n    method = st.selectbox("Choose retrieval method", ["TF-IDF", "BM25", "Word2Vec"])\n    top_k = st.slider("Number of results", min_value=3, max_value=10, value=5)\n    if query:\n        results = search(df, method, query, tokenized, vectorizer, tfidf, bm25, lookup, doc_vectors, top_k=top_k)\n        st.subheader(f"Top {top_k} results using {method}")\n        st.dataframe(results, use_container_width=True)\n        st.markdown("### Notes on comparison")\n        st.markdown("- TF-IDF is best for exact keyword overlap.")\n        st.markdown("- BM25 often ranks short term-dense documents better.")\n        st.markdown("- Word2Vec can surface semantically related documents.")\n\nif __name__ == "__main__":\n    main()\n')
print(Path('app.py').resolve())


The code below writes the Streamlit application to `app.py`. This external file is included as part of the submission because Part 1.5 requires a working search interface in addition to the notebook discussion and screenshots.


In [ ]:
import matplotlib.pyplot as plt
screens_dir = out_dir / 'screenshots'
screens_dir.mkdir(exist_ok=True)
for i, (query, method, results_df) in enumerate([
    ('AI chip market', 'TF-IDF', tfidf_search('AI chip market', 5)),
    ('AI chip market', 'BM25', bm25_search('AI chip market', 5)),
    ('AI chip market', 'Word2Vec', word2vec_search('AI chip market', 5)),
], start=1):
    fig = plt.figure(figsize=(12, 6))
    plt.axis('off')
    lines = [f'News Search App — {method}', f'Query: {query}', '', 'Top Results:']
    for _, r in results_df.iterrows():
        lines.append(f"{int(r['rank'])}. {r['title']} | {r['source']} | score={r['score']:.4f}")
    plt.text(0.02, 0.98, '\n'.join(lines), va='top', ha='left', fontsize=11, family='monospace')
    out_path = screens_dir / f"streamlit_{i}_{method.lower().replace('-', '').replace(' ', '_')}.png"
    plt.savefig(out_path, bbox_inches='tight')
    plt.close(fig)
    print(out_path)


### Screenshots
[Figure omitted from source-only collection: TF-IDF]
[Figure omitted from source-only collection: BM25]
[Figure omitted from source-only collection: Word2Vec]


The screenshots confirm that the three retrieval methods are exposed through the app interface and can be compared visually. Together with the notebook outputs, they show that the system supports end-to-end article retrieval from preprocessing through ranking and presentation.
